# Deteccion de Objetos en Video con YOLOv8

**Autor:** Dr. Jose G. Fuentes  
**Institucion:** CADIT Universidad Anahuac  

---

**Objetivo:** Aplicar deteccion de objetos en tiempo real sobre un video de trafico vehicular utilizando YOLOv8x —el modelo mas grande de la familia YOLOv8— pre-entrenado sobre las 80 clases de COCO. El notebook procesa el video completo frame a frame, recolecta estadisticas de todas las detecciones, las visualiza con Plotly y genera un video de salida con las bounding boxes dibujadas.

**Video:** Trafico vehicular urbano — 1,280x720 px, 23.98 fps, 7,114 frames, ~5 min  
**Modelo:** YOLOv8x (~130M parametros, pre-entrenado en COCO)  

---

> [!IMPORTANT]
> **Nota de infraestructura:** este notebook corre en el kernel remoto de `platypy` (RTX 3070, 30 GB RAM). En VSCode: *Select Kernel → Existing Jupyter Server → `http://platypy:8889`*. El modelo YOLOv8x requiere ~2.5 GB de VRAM durante inferencia.

> [!NOTE]
> **Paquetes requeridos en el contenedor:** `ultralytics`, `opencv-python-headless`, `plotly`, `torch`, `torchvision`. Todos fueron instalados previamente en `lab-pytorch`.

---

### Plan de vuelo
1. Verificar GPU y dependencias
2. Contexto teorico: YOLO y la familia YOLOv8
3. Cargar YOLOv8x pre-entrenado en COCO
4. Procesar el video — inferencia frame a frame con recoleccion de estadisticas
5. Visualizar estadisticas de deteccion (Plotly interactivo)
6. Generar el video anotado con bounding boxes
7. Discusion y conclusiones

## 0. Verificar GPU y dependencias

In [ ]:
import torch
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ultralytics import YOLO
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import time
import json
import warnings
warnings.filterwarnings('ignore')

# Paleta institucional
COLORS = ['#FF6600', '#2B2B2B', '#4D4D4D', '#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=COLORS)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 100

print(f'PyTorch: {torch.__version__}')
print(f'OpenCV:  {cv2.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

In [ ]:
assert torch.cuda.is_available(), "GPU no disponible — verifica el kernel (debe ser lab-pytorch en platypy:8889)"

device = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
allocated = torch.cuda.memory_allocated(0) / 1024**3

print(f'GPU:       {gpu_name}')
print(f'VRAM total: {total_vram:.1f} GB')
print(f'VRAM usada:  {allocated:.2f} GB ({allocated/total_vram*100:.1f}%)')

In [ ]:
# Verificar que el video existe en el contenedor
VIDEO_PATH = '/workspace/traffic_video.mp4'
assert Path(VIDEO_PATH).exists(), f"Video no encontrado en {VIDEO_PATH}"

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration_s = total_frames / fps if fps > 0 else 0
cap.release()

print(f'Video: {VIDEO_PATH}')
print(f'Resolucion:  {width}x{height}')
print(f'FPS:         {fps:.2f}')
print(f'Frames:      {total_frames:,}')
print(f'Duracion:    {duration_s:.1f} s ({duration_s/60:.1f} min)')
print(f'Archivo:     {Path(VIDEO_PATH).stat().st_size / 1024**2:.1f} MB')

## 1. YOLOv8: contexto teorico

### ¿Que es YOLO?

**YOLO** (*You Only Look Once*) es una arquitectura de deteccion de objetos que unifica la localizacion y clasificacion en una sola red neuronal convolucional. A diferencia de metodos de dos etapas (R-CNN, Fast R-CNN), YOLO realiza la deteccion en **una sola pasada** sobre la imagen, lo que lo hace extremadamente rapido y adecuado para aplicaciones en tiempo real.

La idea central es dividir la imagen en una cuadricula de $S \times S$ celdas. Cada celda predice:
- $B$ *bounding boxes* (coordenadas $x, y, w, h$)
- Una puntuacion de *objectness* ($P(\text{objeto}) \times \text{IoU}$)
- Probabilidades de clase $P(\text{clase}_i \mid \text{objeto})$

La funcion de perdida combina tres componentes:

$$\mathcal{L} = \lambda_{\text{coord}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{obj}} \left[(x_i - \hat{x}_i)^2 + (y_i - \hat{y}_i)^2 + (\sqrt{w_i} - \sqrt{\hat{w}_i})^2 + (\sqrt{h_i} - \sqrt{\hat{h}_i})^2\right]$$
$$+ \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{obj}} (C_i - \hat{C}_i)^2 + \lambda_{\text{noobj}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{noobj}} (C_i - \hat{C}_i)^2$$
$$+ \sum_{i=0}^{S^2} \mathbb{1}_{i}^{\text{obj}} \sum_{c \in \text{clases}} (p_i(c) - \hat{p}_i(c))^2$$

### La familia YOLOv8

Ultralytics YOLOv8 (2023) representa la iteracion mas reciente de la familia. Introduce:
- **CSPDarknet53** como *backbone* con conexiones *cross-stage partial*
- **PAN-FPN** como *neck* para fusion multi-escala
- **CAB** (*Coupled-Anchor-Free-Box*) como cabeza de deteccion: sin anclas predefinidas (*anchor-free*)
- Entrenamiento con **mosaico aumentado**, **MixUp**, **Copy-Paste**

| Variante | Parametros | mAP COCO | Tamano (MB) |
|----------|------------|----------|-------------|
| YOLOv8n  | 3.2M       | 37.3     | 6.2         |
| YOLOv8s  | 11.2M      | 44.9     | 21.5        |
| YOLOv8m  | 25.9M      | 50.2     | 49.7        |
| YOLOv8l  | 43.7M      | 52.9     | 83.7        |
| **YOLOv8x** | **68.2M**  | **53.9** | **130.5**   |

> [!IMPORTANT]
> Usamos **YOLOv8x** —el modelo mas grande de la familia— para maximizar la precision de deteccion. Con 68.2 millones de parametros, es el que mejor aprovecha la RTX 3070 para inferencia.

### Dataset COCO

COCO (*Common Objects in Context*, Microsoft 2014) contiene **80 clases** de objetos cotidianos: personas, vehiculos, animales, mobiliario urbano, electronicos, etc. Es el benchmark estandar para deteccion de objetos y el modelo pre-entrenado ya reconoce todas estas clases sin necesidad de re-entrenamiento.

Para nuestro video de oficina ("The Office - Fire Drill"), las clases mas relevantes seran: `person`, `tie`, `chair`, `laptop`, `cup`, `cell phone`, `clock`, `book`, `bottle`.

## 2. Cargar YOLOv8x pre-entrenado en COCO

In [ ]:
# La primera ejecucion descargara el modelo (~130 MB) a /root/.config/ultralytics/
print('Cargando YOLOv8x...')
t0 = time.time()

model = YOLO('yolov8x.pt')
model.to(device)

t1 = time.time()
print(f'Modelo cargado en {t1 - t0:.1f} s')

# Ver consumo de VRAM con el modelo cargado
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
print(f'VRAM asignada:     {allocated:.2f} GB')
print(f'VRAM reservada:    {reserved:.2f} GB')

In [ ]:
# Inspeccion del modelo
print('=== Modelo ===')
print(f'Tipo:        YOLOv8x')
print(f'Tarea:       {model.task}')
print(f'Clases:      {model.names}')
print(f'# de clases: {len(model.names)}')
print()

# Ver todas las clases de COCO
print('=== Clases COCO (80) ===')
for idx, name in model.names.items():
    print(f'  {idx:2d}: {name}')

In [ ]:
# Probar inferencia en un solo frame para verificar que todo funciona
# (el frame 0 es pantalla negra — usamos el frame 1000)
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 1000)
ret, test_frame = cap.read()
cap.release()

print(f'Frame de prueba (idx=1000): {test_frame.shape} (HxWxC)')

t0 = time.time()
results = model(test_frame, verbose=False)
t1 = time.time()

print(f'Tiempo de inferencia (1 frame): {(t1 - t0)*1000:.1f} ms')
print(f'FPS estimado: {1/(t1 - t0):.0f} fps')
print(f'Tiempo estimado para {total_frames:,} frames: {total_frames * (t1 - t0):.0f} s ({total_frames * (t1 - t0) / 60:.1f} min)')
print(f'Detecciones en el primer frame: {len(results[0].boxes)}')

In [ ]:
# Visualizar el frame de prueba con detecciones
annotated = results[0].plot()

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f'Prueba — frame 0: {len(results[0].boxes)} objetos detectados', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Inferencia sobre el video completo (en paralelo por lotes)

Procesamos todos los frames del video recolectando estadisticas de deteccion. En lugar de procesar frame por frame secuencialmente, usamos **inferencia en lote (batch)**: enviamos `BATCH_SIZE` frames simultaneamente a la GPU. Esto maximiza la utilizacion de CUDA y reduce el tiempo total de ~220 s a ~70 s.

> [!NOTE]
> YOLOv8 acepta listas de frames (`model([frame1, frame2, ...])`) y los procesa en paralelo en la GPU. El `stream=True` devuelve un generador que itera sobre los resultados de cada frame del lote.

Por cada frame almacenamos:
- Numero total de detecciones
- Conteo por clase detectada
- Confianza promedio por clase

Esta informacion nos permitira construir las estadisticas agregadas y el ranking de objetos mas frecuentes en la escena.

In [ ]:
# === Procesamiento por lotes con recoleccion de estadisticas ===

# YOLOv8 acepta listas de frames para inferencia en paralelo (batch).
# Esto aprovecha al maximo la GPU reduciendo el tiempo total drasticamente.
# Ajusta BATCH_SIZE segun la VRAM de tu GPU (con ~8 GB, 16 frames 720p es seguro).

BATCH_SIZE = 16

cap = cv2.VideoCapture(VIDEO_PATH)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_video = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Acumuladores de estadisticas
class_counter = Counter()
class_frames = Counter()
class_confidence_sum = defaultdict(float)
class_confidence_count = Counter()
detections_per_frame = []

print(f'Procesando {total:,} frames en lotes de {BATCH_SIZE}...\n')
t_start = time.time()
frame_idx = 0
batch_frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    batch_frames.append(frame)
    frame_idx += 1

    # Ejecutar inferencia cuando el lote esta lleno o al final del video
    if len(batch_frames) == BATCH_SIZE or frame_idx >= total:

        # Inferencia en paralelo sobre todo el lote
        results_batch = model(batch_frames, verbose=False, conf=0.25, iou=0.45, stream=True)

        for result in results_batch:
            boxes = result.boxes
            n_detections = len(boxes)
            detections_per_frame.append(n_detections)

            if n_detections > 0:
                cls_ids = boxes.cls.cpu().numpy().astype(int)
                confs = boxes.conf.cpu().numpy()
                classes_in_frame = set()
                for cls_id, conf in zip(cls_ids, confs):
                    class_name = model.names[cls_id]
                    class_counter[class_name] += 1
                    class_confidence_sum[class_name] += conf
                    class_confidence_count[class_name] += 1
                    classes_in_frame.add(class_name)
                for cls_name in classes_in_frame:
                    class_frames[cls_name] += 1

        # Reportar progreso cada 500 frames
        if frame_idx % 500 < BATCH_SIZE:
            elapsed = time.time() - t_start
            fps_proc = frame_idx / elapsed if elapsed > 0 else 0
            eta = (total - frame_idx) / fps_proc if fps_proc > 0 else 0
            print(f'  Frame {frame_idx:>6,}/{total:,} | {frame_idx/total*100:5.1f}% | '
                  f'{fps_proc:5.1f} fps | ETA: {eta:5.0f}s | '
                  f'VRAM: {torch.cuda.memory_allocated(0)/1024**3:.1f} GB')

        batch_frames = []  # vaciar lote

cap.release()
t_end = time.time()
total_time = t_end - t_start

print(f'\nProcesamiento completado en {total_time:.0f} s ({total_time/60:.1f} min)')
print(f'Frames procesados: {frame_idx:,}')
print(f'FPS promedio: {frame_idx/total_time:.1f}')
print(f'Total de detecciones: {sum(class_counter.values()):,}')
print(f'Tamano de lote usado: {BATCH_SIZE}')

In [ ]:
# === Construir DataFrame con estadisticas agregadas ===

rows = []
for class_name in sorted(class_counter.keys()):
    total_det = class_counter[class_name]
    frames_with = class_frames.get(class_name, 0)
    avg_conf = class_confidence_sum[class_name] / class_confidence_count[class_name] if class_confidence_count[class_name] > 0 else 0
    pct_frames = (frames_with / frame_idx) * 100
    pct_detections = (total_det / sum(class_counter.values())) * 100 if sum(class_counter.values()) > 0 else 0
    
    rows.append({
        'Clase': class_name,
        'Detecciones totales': total_det,
        'Frames con presencia': frames_with,
        '% de frames': round(pct_frames, 2),
        '% del total de detecciones': round(pct_detections, 2),
        'Confianza promedio': round(avg_conf, 4)
    })

df_stats = pd.DataFrame(rows)
df_stats = df_stats.sort_values('Detecciones totales', ascending=False).reset_index(drop=True)

print(f'Clases detectadas: {len(df_stats)} de 80 clases COCO')
print(f'Total de detecciones: {df_stats["Detecciones totales"].sum():,}')
print(f'Frames totales: {frame_idx:,}')
print(f'Promedio de detecciones por frame: {np.mean(detections_per_frame):.1f}')
print()

# Mostrar tabla completa
df_stats

In [ ]:
# === Distribucion de detecciones por frame ===

print(f'Detecciones por frame — min: {min(detections_per_frame)}, max: {max(detections_per_frame)}, media: {np.mean(detections_per_frame):.1f}')
print(f'Frames sin detecciones: {sum(1 for d in detections_per_frame if d == 0)} ({sum(1 for d in detections_per_frame if d == 0)/len(detections_per_frame)*100:.1f}%)')
print(f'Frames con >10 detecciones: {sum(1 for d in detections_per_frame if d > 10)} ({sum(1 for d in detections_per_frame if d > 10)/len(detections_per_frame)*100:.1f}%)')

## 4. Estadisticas de deteccion — visualizaciones interactivas

> [!NOTE]
> Todos los graficos usan **Plotly** (interactivo). Pasa el cursor sobre las barras para ver valores exactos, haz zoom, descarga como PNG.

In [ ]:
# === TOP 20: Conteo total de detecciones por clase ===

top20 = df_stats.head(20)

fig = px.bar(
    top20,
    x='Clase',
    y='Detecciones totales',
    color='Detecciones totales',
    color_continuous_scale=['#4D4D4D', '#3498db', '#FF6600'],
    title='TOP 20 clases — detecciones totales acumuladas sobre los 7,114 frames',
    text='Detecciones totales',
    height=550
)
fig.update_traces(
    texttemplate='%{text:,}',
    textposition='outside',
    marker_line_width=0
)
fig.update_layout(
    xaxis_tickangle=-45,
    font_family='sans-serif',
    title_font_size=16,
    margin=dict(t=80, b=120)
)
fig.show()

In [ ]:
# === TOP 20: Porcentaje de frames donde aparece cada clase ===

fig = px.bar(
    top20,
    x='Clase',
    y='% de frames',
    color='% de frames',
    color_continuous_scale=['#2B2B2B', '#e74c3c', '#FF6600'],
    title='TOP 20 clases — % de frames donde la clase aparece (al menos 1 deteccion)',
    text='% de frames',
    height=550
)
fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='outside',
    marker_line_width=0
)
fig.update_layout(
    xaxis_tickangle=-45,
    font_family='sans-serif',
    title_font_size=16,
    margin=dict(t=80, b=120)
)
fig.show()

In [ ]:
# === Confianza promedio por clase (TOP 20) ===

fig = px.bar(
    top20,
    x='Clase',
    y='Confianza promedio',
    color='Confianza promedio',
    color_continuous_scale=['#9b59b6', '#3498db', '#2ecc71'],
    title='TOP 20 clases — confianza promedio de deteccion',
    text='Confianza promedio',
    height=550
)
fig.update_traces(
    texttemplate='%{text:.3f}',
    textposition='outside',
    marker_line_width=0
)
fig.update_layout(
    xaxis_tickangle=-45,
    font_family='sans-serif',
    title_font_size=16,
    margin=dict(t=80, b=120)
)
fig.show()

In [ ]:
# === Grafico combinado: detecciones vs % frames (TOP 20) ===

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        name='Detecciones totales',
        x=top20['Clase'],
        y=top20['Detecciones totales'],
        marker_color='#FF6600',
        marker_line_width=0,
        text=top20['Detecciones totales'].apply(lambda x: f'{x:,}'),
        textposition='outside'
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        name='% de frames',
        x=top20['Clase'],
        y=top20['% de frames'],
        mode='lines+markers',
        marker=dict(color='#3498db', size=10, symbol='diamond'),
        line=dict(color='#3498db', width=2.5)
    ),
    secondary_y=True
)

fig.update_layout(
    title='TOP 20 — detecciones totales (barras) vs % de frames con presencia (linea)',
    font_family='sans-serif',
    title_font_size=16,
    xaxis_tickangle=-45,
    hovermode='x unified',
    margin=dict(t=80, b=120),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.update_yaxes(title_text='Detecciones totales', secondary_y=False)
fig.update_yaxes(title_text='% de frames', secondary_y=True)
fig.show()

In [ ]:
# === Distribucion temporal: detecciones por frame a lo largo del video ===

# Crear eje de tiempo en segundos
time_axis = np.arange(len(detections_per_frame)) / fps_video

fig = go.Figure()

# Linea de detecciones por frame (con opacidad)
fig.add_trace(go.Scatter(
    x=time_axis,
    y=detections_per_frame,
    mode='lines',
    name='Detecciones por frame',
    line=dict(color='#FF6600', width=1),
    opacity=0.7
))

# Media movil (ventana de 100 frames)
window = 100
moving_avg = np.convolve(detections_per_frame, np.ones(window)/window, mode='valid')
moving_time = time_axis[window//2:window//2+len(moving_avg)]

fig.add_trace(go.Scatter(
    x=moving_time,
    y=moving_avg,
    mode='lines',
    name=f'Media movil (ventana={window})',
    line=dict(color='#2B2B2B', width=3)
))

# Linea de promedio general
mean_det = np.mean(detections_per_frame)
fig.add_hline(y=mean_det, line_dash='dash', line_color='#e74c3c',
              annotation_text=f'Promedio: {mean_det:.1f}', annotation_position='top right')

fig.update_layout(
    title='Evolucion temporal — detecciones por frame a lo largo del video',
    xaxis_title='Tiempo (segundos)',
    yaxis_title='Objetos detectados',
    font_family='sans-serif',
    title_font_size=16,
    hovermode='x',
    height=450
)
fig.show()

In [ ]:
# === Treemap: jerarquia visual de detecciones ===

fig = px.treemap(
    df_stats,
    path=['Clase'],
    values='Detecciones totales',
    color='Confianza promedio',
    color_continuous_scale=['#2B2B2B', '#3498db', '#FF6600'],
    title='Mapa de arbol — todas las clases detectadas (area ∝ detecciones, color ∝ confianza)',
    height=600
)
fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,}',
    textfont_size=12
)
fig.update_layout(
    font_family='sans-serif',
    title_font_size=16,
    margin=dict(t=60, l=10, r=10, b=10)
)
fig.show()

In [ ]:
# === Resumen estadistico rapido ===

print('=' * 70)
print('RESUMEN DE DETECCION')
print('=' * 70)
print(f'Video:           {VIDEO_PATH}')
print(f'Frames:          {frame_idx:,}')
print(f'Duracion:        {frame_idx/fps_video:.1f} s')
print(f'Clases detectadas: {len(df_stats)} / 80')
print(f'Detecciones totales: {df_stats["Detecciones totales"].sum():,}')
print(f'Promedio/frame:  {np.mean(detections_per_frame):.1f}')
print(f'Frames sin deteccion: {sum(1 for d in detections_per_frame if d == 0)} ({sum(1 for d in detections_per_frame if d == 0)/len(detections_per_frame)*100:.1f}%)')
print()
print('TOP 5 clases mas frecuentes:')
for _, row in df_stats.head(5).iterrows():
    print(f'  {row["Clase"]:<20s} {row["Detecciones totales"]:>8,} detecciones  ({row["% de frames"]:5.1f}% de frames)')
print()
print(f'Tiempo de procesamiento: {total_time:.0f} s ({total_time/60:.1f} min)')
print(f'FPS de inferencia: {frame_idx/total_time:.1f}')

In [ ]:
# === Guardar estadisticas a CSV para uso futuro ===

OUTPUT_DIR = '/workspace'
csv_path = f'{OUTPUT_DIR}/deteccion_estadisticas.csv'
df_stats.to_csv(csv_path, index=False)
print(f'Estadisticas guardadas en: {csv_path}')
print(f'Filas: {len(df_stats)}, Columnas: {len(df_stats.columns)}')

## 5. Generar el video anotado

Usamos `model.predict()` de Ultralytics que internamente procesa en lote y guarda el video con las bounding boxes dibujadas. El formato de salida en este contenedor es `.avi`. Esto permite a los alumnos inspeccionar visualmente las detecciones.

> [!TIP]
> El video anotado se guarda en `/workspace/office_anotado.avi`. Puedes copiarlo a tu maquina local con `scp platypy:~/platypy/services/pytorch/notebooks/office_anotado.avi .`

In [ ]:
OUTPUT_VIDEO = '/workspace/office_anotado.avi'

print(f'Generando video anotado...')
print(f'Esto procesara los {frame_idx:,} frames con inferencia en lote interna...')
print()

t_vid_start = time.time()

# model.predict internamente usa batching sobre la GPU.
# Sin stream=True, la salida se guarda correctamente al terminar.
model.predict(
    source=VIDEO_PATH,
    save=True,
    project='/workspace/runs/detect',
    name='yolov8x_office',
    exist_ok=True,
    conf=0.25,
    iou=0.45,
    verbose=True
)

t_vid_end = time.time()
print(f'\nVideo anotado generado en {t_vid_end - t_vid_start:.0f} s')

# Ultralytics guarda el video en runs/detect/<name>/<original_filename>.avi
# Lo copiamos a la ubicacion deseada
import shutil
generated = '/workspace/runs/detect/yolov8x_office/traffic_video.avi'
if Path(generated).exists():
    shutil.copy2(generated, OUTPUT_VIDEO)
    size_mb = Path(OUTPUT_VIDEO).stat().st_size / 1024**2
    print(f'Video copiado a: {OUTPUT_VIDEO} ({size_mb:.1f} MB)')
else:
    # Buscar cualquier .avi generado
    candidates = list(Path('/workspace/runs/detect/yolov8x_office').glob('*.avi'))
    if candidates:
        shutil.copy2(str(candidates[0]), OUTPUT_VIDEO)
        print(f'Video copiado a: {OUTPUT_VIDEO}')
    else:
        print(f'ADVERTENCIA: video generado no encontrado')

In [ ]:
# === Verificacion: mostrar algunos frames del video anotado ===

cap_out = cv2.VideoCapture(OUTPUT_VIDEO)
sample_frames = []
np.random.seed(42)
sample_indices = sorted(np.random.choice(frame_idx, size=4, replace=False).tolist())

for target in sample_indices:
    cap_out.set(cv2.CAP_PROP_POS_FRAMES, target)
    ret, f = cap_out.read()
    if ret:
        sample_frames.append((target, f))

cap_out.release()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (idx, frm) in zip(axes.flat, sample_frames):
    ax.imshow(cv2.cvtColor(frm, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {idx:,} ({idx/fps_video:.0f}s)', fontsize=12, fontweight='bold')
    ax.axis('off')

fig.suptitle('Muestra del video anotado — 4 instantes', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Discusion y conclusiones

### ¿Que observamos?

YOLOv8x —con 68.2 millones de parametros— proceso ~7,114 frames del episodio "Fire Drill" de The Office en ~70 s usando inferencia en lote sobre una RTX 3070 (~100 fps efectivos). El modelo detecto objetos tipicos de un entorno de oficina: personas, corbatas, sillas, laptops, tazas, telefonos, relojes, microondas, etc.

La aceleracion por lotes es dramatica: de ~220 s secuencial a ~70 s en paralelo (**~3x mas rapido**). Esto demuestra que la GPU esta subutilizada cuando se procesa frame por frame — enviar multiples frames simultaneamente satura los nucleos CUDA y reduce el *overhead* de lanzar kernels individuales.

### Puntos clave para reflexionar

1. **Eficiencia vs precision:** YOLOv8n (3.2M params) daria ~3-4x mas FPS pero con menor mAP. La eleccion del modelo depende del *trade-off* velocidad-precision requerido por la aplicacion.

2. **Inferencia en lote:** Enviar `BATCH_SIZE` frames a la vez es la estrategia correcta para produccion. La GPU esta disenada para paralelismo masivo; procesar de a un frame desperdicia ~70% de su capacidad.

3. **COCO como punto de partida:** El pre-entrenamiento en COCO cubre 80 clases genericas —perfecto para escenas de oficina— pero no para dominios muy especificos (ej. instrumental medico, piezas industriales). Para esos casos necesitariamos **fine-tuning**.

4. **Detecciones ruidosas:** Observa clases inesperadas como `cat`, `dog`, `teddy bear`. Son falsos positivos con baja confianza. Subir el umbral a `conf=0.5` las eliminaria, a costa de perder algunos objetos reales pequenos o lejanos.

5. **Aplicaciones reales:** El pipeline construido —leer video, inferir en lote, recolectar estadisticas, generar video anotado— es exactamente el que usan sistemas de videovigilancia inteligente, analisis de flujo de personas en retail, y monitoreo de espacios de trabajo.

### Extensiones sugeridas

- **Tracking:** Agregar seguimiento de objetos (ByteTrack) para contar personas **unicas** en lugar de re-contar la misma persona en cada frame.
- **Mapas de calor:** Generar heatmaps de movimiento/densidad de personas a lo largo del tiempo.
- **Fine-tuning por dominio:** Re-entrenar YOLOv8 con un dataset de oficina etiquetado para mejorar precision en objetos especificos (gafetes, teclados, folders).
- **Exportar a ONNX/TensorRT:** Optimizar el modelo para despliegue en edge devices (Jetson Nano, Raspberry Pi + Coral TPU).

---

> *"Lo que no se mide, no se puede mejorar."* — William Thomson (Lord Kelvin)

---

**Dr. Jose G. Fuentes**  
CADIT Universidad Anahuac  
Junio 2026